# Semana 1 — Verificación del entorno y primera escena 3D

**Electiva Profesional I (El Metaverso) · 2026-II**

Este cuaderno hace dos cosas:

1. **Comprueba que su entorno `mv2026` funciona** (conda, Python, numpy, pythreejs).
2. Le permite **recorrer, sin nombrarlas todavía, las primeras etapas de construir un entorno inmersivo** — y a enfrentar situaciones como una mala rotación rotación mal ajjstada (grados vs. radianes) que deja el mundo en una posición absurda **sin que Python lance ningún error**.

El mundo es neutral y genérico: **un planeta y unas lunas** que orbitan. Sirve porque en pocas líneas deja ver cómo se **modela** un mundo con coordenadas y cómo la **honestidad numérica** decide si la escena está bien o solo se ve bien.

**Cómo trabajar:** ejecute las celdas **de arriba abajo**. Donde diga `# TODO`, escriba su respuesta. Al final, `Kernel → Restart & Run All` y guarde con las salidas. Ese `.ipynb` ejecutado es el **Entregable B **.

> No borre ningún error. Si una celda falla, **pegue el mensaje** en una celda de texto debajo y siga. Documentar el error puntúa.

## 1. ¿Está vivo el entorno?

Si esta celda corre sin error y la ruta menciona `mv2026`, su entorno está bien montado.

In [2]:
import sys, platform
print('Python  :', platform.python_version())
print('Ejecutable (debe mencionar mv2026):')
print('  ', sys.executable)

import numpy as np
print('numpy   :', np.__version__)

# pythreejs es opcional para el render 3D. Si no carga, el cuaderno sigue funcionando
# (la parte numerica -que es el corazon de hoy- no depende de el).
try:
    import pythreejs as p3
    TENGO_3D = True
    print('pythreejs: OK (podrá ver la escena 3D mas abajo)')
except Exception as e:
    TENGO_3D = False
    print('pythreejs: no disponible ->', e)
    print('  No pasa nada: la parte numerica funciona igual. El render 3D es un extra.')

print('\nSi llegaste hasta aqui sin error rojo, el entorno mv2026 funciona.')

Python  : 3.14.6
Ejecutable (debe mencionar mv2026):
   C:\Users\loghan\miniconda3\envs\mv2026\python.exe
numpy   : 2.5.1
pythreejs: OK (podrá ver la escena 3D mas abajo)

Si llegaste hasta aqui sin error rojo, el entorno mv2026 funciona.


## 2. `[ETAPA ?]` — Antes de tocar nada: ¿Qué experiencia queremos?

Vamos a construir un pedacito de mundo: **un planeta con lunas que lo orbitan**. Alguien entra y lo mira girar. Parece un adorno, pero para construirlo hay que decidir *para qué* existe.

**`# TODO 1`** — En una frase, ¿qué **experiencia** o **necesidad** podría justificar un mundo así? (Ejemplos para pensar, no para copiar: enseñar a un niño cómo orbitan los cuerpos; ambientar la sala de espera de un museo; un fondo interactivo para una app. Reemplace esta línea con su idea.)

In [ ]:
#Una herramienta didáctica orientada a desglosar el proceso de creación 3D

## 3. `[ETAPA ?]` — Modelar el mundo con coordenadas

Un punto en 3D es `(x, y, z)`: `x` izquierda-derecha, `y` abajo-arriba, `z` adelante-atras.

Ponemos el **planeta** en el centro `(0, 0, 0)`. Las **lunas** van sobre un circulo de radio `R` alrededor del planeta, cada una en un **angulo** distinto.

In [4]:
centro = np.array([0.0, 0.0, 0.0])   # el planeta, en el origen
R = 5.0                              # radio de la orbita

# Queremos colocar una luna dado un angulo. En el plano x-z:
#   angulo 0    -> justo enfrente del planeta   -> (0, 0, R)
#   angulo 90   -> a un lado                     -> (R, 0, 0)
#   angulo 180  -> justo detras                  -> (0, 0, -R)
# Esa es nuestra intuicion. Vamos a programarla.
print('Planeta en:', centro, '| radio de orbita:', R)

Planeta en: [0. 0. 0.] | radio de orbita: 5.0


## 4. La colocación preliminar

Escribimos la función suministrando el ángulo y al ejecutarla debe arrojarme la posición.

In [8]:
def posicion_luna_preliminar(angulo, radio=R):
    x = radio * np.sin(angulo)
    z = radio * np.cos(angulo)
    return np.array([x, 0.0, z]).round(2)

for a in [0, 90, 180, 270]:
    print(f'angulo {a:>3} -> {posicion_luna_preliminar(a)}')

angulo   0 -> [0. 0. 5.]
angulo  90 -> [ 4.47  0.   -2.24]
angulo 180 -> [-4.01  0.   -2.99]
angulo 270 -> [-0.88  0.    4.92]


**Mire los resultados y compárelos con la intuición de la sección 3:**

- `0`   debería dar `(0, 0, 5)` ... y da `(0, 0, 5)`. Bien (por suerte).
- `90`  debería dar `(5, 0, 0)` (a un lado) ... **pero NO lo da.**
- `180` debería dar `(0, 0, -5)` (detrás) ... **tampoco.**

**No dio ningún error, pero resultó en una escena mal armada.** Con dos lunas se nota; si tuvieramos cientos de objetos, se habría creado una escena errada como si estuviera bien diseñada.

**`# TODO 2`** — Mire la posición del ángulo `90` ¿Por qué cree que quedó en el lugar equivocado, si el código "se ve" correcto? (¿Será necesario reemplazar esa línea?)

## 5. `[ETAPA ?]` — Mirar de cerca qué pasó

`numpy` no utiliza grados sexagesimales: emplea **radianes**. Cuando le suministramos el valor `90`, no entiende "90 grados", sino "90 radianes", que equivale a vuelta y media... varias veces.

In [9]:
print('np.cos(90)            =', round(np.cos(90), 4), '  <- numpy leyo 90 RADIANES')
print('np.cos(np.radians(90))=', round(np.cos(np.radians(90)), 4), '  <- 90 grados de verdad (~0)')
print()
print('Cuando numpy recibe 90 y cree que son radianes, en grados eso es:',
      round(np.degrees(90), 0), 'grados')
print('...es decir, mas de 14 vueltas completas. Por eso la luna aterriza en cualquier lado.')

np.cos(90)            = -0.4481   <- numpy leyo 90 RADIANES
np.cos(np.radians(90))= 0.0   <- 90 grados de verdad (~0)

Cuando numpy recibe 90 y cree que son radianes, en grados eso es: 5157.0 grados
...es decir, mas de 14 vueltas completas. Por eso la luna aterriza en cualquier lado.


**`# TODO 3`** — Con sus palabras: ¿Qué error se cometió en este caso? ¿En qué unidad esperaba `numpy` el ángulo y cuál fue la que le dimos? (Reemplace esta línea.)

In [ ]:
# El error cometido fue al momento de intentar utilizar grados hexadecimales ya que numpy utiliza o esperaba grados en radianes, por tal razon la luna queda ubicada en coordenadas incorrectas

## 6. `[ETAPA ?]` — Simular la dinámica, decidiendo

Cada línea de abajo es **una decisión**, no una receta. La principal: *"la entrada viene en grados; la convierto a radianes antes de dársela a numpy"*.

In [5]:
def posicion_luna(angulo_grados, radio=R):
    a = np.radians(angulo_grados)        # DECISION: la entrada esta en GRADOS -> convertir
    x = radio * np.sin(a)
    z = radio * np.cos(a)
    return np.array([x, 0.0, z]).round(2)

print('Ahora si, con la correccion:')
for a in [0, 90, 180, 270]:
    print(f'angulo {a:>3} -> {posicion_luna(a)}')

# Una simulacion diminuta: una luna que avanza 45 grados por "paso de tiempo".
print('\nOrbita de una luna (un paso cada 45 grados):')
for t, ang in enumerate(range(0, 360, 45)):
    print(f'  paso {t}: angulo {ang:>3} -> {posicion_luna(ang)}')

Ahora si, con la correccion:
angulo   0 -> [0. 0. 5.]
angulo  90 -> [5. 0. 0.]
angulo 180 -> [ 0.  0. -5.]
angulo 270 -> [-5.  0. -0.]

Orbita de una luna (un paso cada 45 grados):
  paso 0: angulo   0 -> [0. 0. 5.]
  paso 1: angulo  45 -> [3.54 0.   3.54]
  paso 2: angulo  90 -> [5. 0. 0.]
  paso 3: angulo 135 -> [ 3.54  0.   -3.54]
  paso 4: angulo 180 -> [ 0.  0. -5.]
  paso 5: angulo 225 -> [-3.54  0.   -3.54]
  paso 6: angulo 270 -> [-5.  0. -0.]
  paso 7: angulo 315 -> [-3.54  0.    3.54]


## 7. `[ETAPA ?]` — Ver el mundo en 3D (extra con pythreejs)

Si `pythreejs` cargó, esta celda dibuja el planeta y cuatro lunas **ya bien colocadas**. Si no cargó, no pasa nada: la escena ya quedó bien definida por las coordenadas de arriba, que es lo que importa hoy.

In [6]:
if TENGO_3D:
    planeta = p3.Mesh(
        geometry=p3.SphereGeometry(radius=1.2, widthSegments=24, heightSegments=24),
        material=p3.MeshStandardMaterial(color='#3b7dd8'),
        position=[0, 0, 0])

    lunas = []
    for ang in [0, 90, 180, 270]:
        pos = posicion_luna(ang).tolist()
        lunas.append(p3.Mesh(
            geometry=p3.SphereGeometry(radius=0.4, widthSegments=16, heightSegments=16),
            material=p3.MeshStandardMaterial(color='#d8b23b'),
            position=pos))

    camara = p3.PerspectiveCamera(position=[8, 6, 12], fov=50)
    luz = p3.DirectionalLight(color='white', intensity=1.0, position=[5, 8, 5])
    ambiente = p3.AmbientLight(intensity=0.4)
    escena = p3.Scene(children=[planeta, *lunas, camara, luz, ambiente])
    render = p3.Renderer(scene=escena, camera=camara,
                         controls=[p3.OrbitControls(controlling=camara)],
                         width=520, height=360)
    display(render)
else:
    print('pythreejs no esta disponible en este equipo.')
    print('La escena, de todos modos, ya quedo bien definida por coordenadas:')
    for ang in [0, 90, 180, 270]:
        print(f'  luna {ang:>3} grados -> {posicion_luna(ang)}')

Renderer(camera=PerspectiveCamera(position=(8.0, 6.0, 12.0), projectionMatrix=(1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0…

## 8. `[ETAPA ?]` — Publicar y comunicar

Hoy no publicamos el mundo en internet todavía (eso es WebXR/glTF, **semana 14**). Pero sí lo comunicamos.

**`# TODO 4`** — En una frase que entienda alguien **no técnico**, describa la experiencia que construyó: qué es, qué se ve y para qué podría servir. Esa frase es la *clave* de su mundo.

---

## Cierre — nombre lo que hizo

Acaba de recorrer las primeras etapas del **proceso de construir un entorno inmersivo**. Cada sección llevaba `[ETAPA ?]`. Ahora póngales nombre con sus palabras:

| Sección | `# TODO 5` — ¿Qué etapa fue? (Su nombre) |
|---|---|
| 2. ¿Qué experiencia queremos? | Pregunta guia | 
| 3. Modelar con coordenadas | Identificar coordenadas |
| 5. Mirar de cerca qué pasó | Identificar errores |
| 6. Simular la dinámica, decidiendo | Corregir errores |
| 8. Publicar y comunicar | |

> Faltan **dos** etapas que **no** hicimos hoy: dejar que el usuario **interactúe de verdad** (moverse, tocar, un avatar) e inlcuir un **componente de inteligencia/datos** que reaccione. Esas son las **Unidades 2 y 3**. Hoy las dejamos anotadas y nada más.

> **La lección que sostiene el curso:** el código "se veía" bien y estaba mal, y Python **no** avisó. En un mundo 3D, muchísimos errores son así —una unidad equivocada, un eje cambiado, un orden de transformaciones— y el código no arroja ningún error: solo se ven raros, o ni eso. Por eso la **honestidad numérica** (verificar contra lo que uno espera, con una línea base simple) es tan importante como que la escena "se vea bonita". Volveremos a ella en la semana 7 (línea base y verificación de la simulación).